---
title: "Chat with Parliament"
author: "Chresten R. Søndergaard"
title-block-banner: true
title-block-style: default
date: "2026-02-27"
abstract: >
    Here we will demonstrate *agentic delegation* where a central AI agent can delegate sub-tasks to other agents for a well-defined division of the work. While more complex than having a single AI agent, delegation can lead to better overall performance.

    Here we set up a political analyst agent to have access to a series of party expert agents (one for each party currently in the Danish parliament). The political analyst takes input from the user about their viewpoints and then queries relevant parties on their opinions in order to rank the parties according to their alignment with the user.
    
    This page is part of the [chat_with_parliament](https://github.com/crs17/chat_with_parliament) repository which contains all the code for the demo.
---

# Chat with Parliament
Here we will demonstrate a multi-agent AI implementation. A political analyst agent takes input from users about their political views and which subjects matter to them.

For each party in the Danish parliament, a party expert agent is implemented with access to a vector database collection containing the relevant party's manifesto. Under the hood there is actually only one party expert agent, it just filters the vector database according to which party it is asked about.

In an agentic delegation setup, the political analyst can choose to query one or more of the party expert agents on specific subjects. After having gathered the relevant responses from the party experts, the political analyst agent can score each of the parties in relation to their alignment with the user's input.

![Architecture](static/images/chat_with_parliament.drawio.svg)


## Querying the Political Analyst Agent
Let's start by running the setup with two statements in opposite directions valuing either climate change concerns or economic growth the highest.

### Opinion: "Klimaet er vigtigere end erhvervslivet."

Below we see the logs of the model run showing how the political analyst agent queries the individual party expert agents, the responses being sent back, and finally how political analyst compiles the answers. Further below we see the scores given to the individual parties by the political analyst. 

In [ ]:
!make setup
!make fetch_models
!make run
!make logfire_auth
!make populate_db

In [3]:
from backend.agents import political_analyst_agent


def print_party_scores(result):
    print('Party Score Reason')
    for recommendation in sorted(result.output.results, key=lambda x: x.score, reverse=True):
        print(f'{recommendation.party_id:5} {recommendation.score:5} {recommendation.reason}')


Logfire project URL: ]8;id=232784;https://logfire-eu.pydantic.dev/crs17/chat-with-parliament\https://logfire-eu.pydantic.dev/crs17/chat-with-parliament]8;;\

In [4]:

result_klima_1 = await political_analyst_agent.run(
    "Klimaet er vigtigere end erhvervslivet! Lad mig se en score for alle partierne"
)

14:14:40.762 political_analyst_agent run
14:14:40.769   chat claude-sonnet-4-5
14:14:49.016   running 12 tools
14:14:49.017     running tool: consult_party_expert

>>> Analyst → Party Expert: Socialdemokratiet (A), query='klimapolitik og prioritering af klima over erhvervsinteresser'
14:14:49.018       party_expert_agent run
14:14:49.018     running tool: consult_party_expert

>>> Analyst → Party Expert: Radikale Venstre (B), query='klimapolitik og prioritering af klima over erhvervsinteresser'
14:14:49.018       party_expert_agent run
14:14:49.019     running tool: consult_party_expert

>>> Analyst → Party Expert: Det Konservative Folkeparti (C), query='klimapolitik og prioritering af klima over erhvervsinteresser'
14:14:49.019       party_expert_agent run
14:14:49.019     running tool: consult_party_expert

>>> Analyst → Party Expert: SF - Socialistisk Folkeparti (F), query='klimapolitik og prioritering af klima over erhvervsinteresser'
14:14:49.020       party_expert_agent run
14:14

In [5]:
print_party_scores(result_klima_1)

Party Score Reason
Ø        10 Enhedslisten er det mest radikale parti på klima. De kæmper for 'grøn omstilling - for mennesker, ikke for profit', vil have høje CO₂-afgifter der rammer de største udledere hårdest, og afviser grønne gimmicks. De vil have Danmark klimaneutral senest 2040 og fri af fossile brændsler i 2030. Klart prioriterer klima over erhvervsinteresser.
Å         9 Alternativet siger direkte, at klimaloven skal prioritere klima og fremtidige generationers rettigheder - IKKE erhvervslivets vækstpotentiale. De vil have Danmarks udledninger ned til nul mellem 2025-2035, som videnskaben kræver. Meget ambitiøs klimapolitik der klart går foran erhvervsinteresser.
F         7 SF ser den grønne omstilling som 'helt afgørende' og vil have Danmark til at gå forrest i klimakampen med ambitiøs klimapolitik. De kombinerer klima med erhverv, men fremhæver klart klimaets vigtighed og vil leve op til Parisaftalen. Stærk klimaprofil, selvom de ikke direkte siger klima over erhverv.
B   

These scores do align pretty well with how I would have assigned them. Intuitively, the "climate change concerns above economic growth" theme should resonate better with the left-wing parties than the right-wing parties.

### Opinion: "Økonomisk vækst er førsteprioritet."
Next, we try roughly the opposite opinion, namely that economic growth is the top priority. And as one would hope, the parties are now ranked in roughly the opposite direction.

In [7]:
result_klima_2 = await political_analyst_agent.run(
    "Økonomisk vækst er førsteprioritet for mig. Lad mig se en score for alle partierne"
)

14:17:49.636 political_analyst_agent run
14:17:49.639   chat claude-sonnet-4-5
14:17:57.530   running 12 tools
14:17:57.531     running tool: consult_party_expert

>>> Analyst → Party Expert: Socialdemokratiet (A), query='økonomisk vækst'
14:17:57.532       party_expert_agent run
14:17:57.533     running tool: consult_party_expert

>>> Analyst → Party Expert: Radikale Venstre (B), query='økonomisk vækst'
14:17:57.534       party_expert_agent run
14:17:57.534     running tool: consult_party_expert

>>> Analyst → Party Expert: Det Konservative Folkeparti (C), query='økonomisk vækst'
14:17:57.548       party_expert_agent run
14:17:57.548     running tool: consult_party_expert

>>> Analyst → Party Expert: SF - Socialistisk Folkeparti (F), query='økonomisk vækst'
14:17:57.549       party_expert_agent run
14:17:57.549     running tool: consult_party_expert

>>> Analyst → Party Expert: Borgernes Parti - Lars Boje Mathiesen (H), query='økonomisk vækst'
14:17:57.549       party_expert_agent run

In [8]:
print_party_scores(result_klima_2)

Party Score Reason
I         9 Liberal Alliance har den mest eksplicitte vækstdagsorden med fokus på skattelettelser, bedre vilkår for virksomheder og investeringer. De argumenterer direkte for, at lavere skatter vil øge beskæftigelse og innovation, hvilket er kernen i klassisk vækstpolitik.
V         9 Venstre sætter vækst og velstand som forudsætning for velfærd. De har konkret løftet arbejdsudbuddet med 30.000+ gennem reformer og prioriterer en sund privat sektor med virkelyst. Klar vækstorienteret tilgang.
H         8 Borgernes Parti ser økonomisk vækst som direkte resultat af et stærkt erhvervsliv. De vil fremme vækst gennem lavere skatter, mindre bureaukrati og bedre iværksættervilkår - en klassisk liberal vækststrategi.
C         7 Konservative har ingen specifik information om vækst i deres program, men partiet er traditionelt erhvervsvenligt og vækstorienteret. Score baseret på partiets generelle profil.
M         6 Moderaterne har ingen specifik information om vækst i deres p

## Technical considerations
During the implementation of this demo, I stumbled on a few technical issues that I had to sort out. I just added them here for clarity.

### Defensive web sites
Some party web sites are very defensive when it comes to allowing automated requests (looking at you, Socialdemokratiet). So, simply calling `requests.get()` did not work. In order to even read the html of some party web sites, both cookies and javascript was needed. Playwright with the Chromium rendering engine came to the rescue.

### Debugging multi-agent setups
With multiple agents complexity can grow rapidly and figuring out exactly where things go wrong can be tedious. Logfire is a powerful tool for debugging multi-agent implementations. In the screenshot below we see a party expert agent formulating a party's stance on "economic growth" based on the relevant chunks from the party manifest.
![Logfire screenshot](static/images/logfire.png)


### Running multiple agents on limited local resources
Multi-agent setups can be more demanding for the LLM models and as everything in this demo runs locally, it was a challenge to find a model capable of dealing with the orchestration of the party experts and still being small enough to run on my 16 GB laptop.
In the end the qwen3:8B model was found to perform reasonably ok while also not crashing my laptop. However evaluation is slow at around 15+ minutes per query and the model really struggled. The model often had problems generating JSON in the required formats and it often also forgot some of the parties in the evaluation.

Switching to Anthropic's API to use Sonnet 4.5 solved these issues. Evaluation times are less than 2 minutes, JSON is on point, all parties are included and results seem intuitively sane.

### Unbalanced amounts of policy data between parties
In this project I just fetched the main page on policy (including subpages) for each party and the amount text available varies quite a bit between parties. That can be seen above where the political analyst agent states that there is no information on climate policy for three parties and that it therefore can not score them. Many other sources could be included to improve the answers but for this demo's purpose I just kept it simple.

## Tech Stack overview
- Playwright for fetching party manifests on party sites that are defensive against automated requests
- Docling for parsing HTML files into markdown
- Chonkie for chunking
- Local Ollama deployment for LLM and embedding models. We do not use docker here as GPU access from docker is not available on Apple Silicon.
- Weaviate for vector database in docker
- Pydantic AI for Agent orchestration
- Logfire for monitoring and debugging of the Agents

## Conclusion
Agentic delegation allows AI agents to tackle large and complex tasks by planning and dividing work into relevant subtasks. There are many plausible commercial use cases for agent delegation. Given the nature of the underlying LLMs these agents especially shine in cases with large amounts of unstructured text. Customer support and experience is one such case including ticket triage, proactive customer outreach and automatic review analysis. Many other use cases exist within marketing, accounting, HR and supply chain automation. 

## How I can help
Please reach out on [LinkedIn](https://www.linkedin.com/in/chresten-søndergaard/) if I can help with multi-agent deployment or other ML/AI endeavors.
